# LeetCode 1162 — As Far from Land as Possible bằng Python

Link đề: https://leetcode.com/problems/as-far-from-land-as-possible/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Breadth-First Search, Matrix

Skeleton LeetCode:

```python
class Solution:
    def maxDistance(self, grid: list[list[int]]) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên** — trong mọi ô nước, lấy khoảng cách **Manhattan** tới ô đất **gần nhất**, rồi trả về giá trị **lớn nhất** trong số đó; nếu lưới toàn đất hoặc toàn nước thì trả `-1`. Điểm cốt lõi: **BFS đa nguồn** (multi-source BFS) xuất phát cùng lúc từ mọi ô đất, hoặc **quy hoạch động hai lượt quét**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: khoảng cách Manhattan `|x0 - x1| + |y0 - y1|`, "max của min", và hai trường hợp trả `-1`.
2. Nắm công cụ: duyệt lưới 4 hướng, `deque`, **BFS đa nguồn**, quy hoạch động (DP) hai lượt quét trên ma trận.
3. Đi từ **vét cạn mọi cặp (nước, đất)** → **BFS từ từng ô nước** → **BFS đa nguồn** → **DP hai lượt**.
4. Hiểu vì sao trên lưới **không vật cản**, khoảng cách BFS 4 hướng đúng bằng khoảng cách Manhattan.
5. Tránh lỗi: quên trường hợp `-1`, BFS từ từng ô nước (quá chậm), đánh dấu thăm muộn gây trùng lặp, DP một lượt.

## 2. Đề bài & ví dụ

Cho lưới `grid` kích thước `n × n` chỉ gồm `0` (nước) và `1` (đất). Tìm một **ô nước** sao cho khoảng cách từ nó tới **ô đất gần nhất** là **lớn nhất**, và trả về khoảng cách đó. Nếu lưới **không có đất** hoặc **không có nước**, trả về `-1`.

Khoảng cách dùng là **Manhattan**: giữa `(x0, y0)` và `(x1, y1)` là `|x0 - x1| + |y0 - y1|`.

**Ví dụ 1:** `grid = [[1,0,1],[0,0,0],[1,0,1]]` → `2` (ô `(1,1)` cách mọi góc đất 2).

**Ví dụ 2:** `grid = [[1,0,0],[0,0,0],[0,0,0]]` → `4` (ô `(2,2)` cách `(0,0)` là 4).

**Ràng buộc:** `n == grid.length == grid[i].length`; `1 <= n <= 100`; `grid[i][j]` là `0` hoặc `1`.

In [1]:
import copy
import random
import time
from collections import deque
from typing import List

examples = [
    ([[1, 0, 1], [0, 0, 0], [1, 0, 1]], 2),
    ([[1, 0, 0], [0, 0, 0], [0, 0, 0]], 4),
    ([[1, 1], [1, 1]], -1),     # toàn đất
    ([[0, 0], [0, 0]], -1),     # toàn nước
]
for g, want in examples:
    print(g, "->", want)

[[1, 0, 1], [0, 0, 0], [1, 0, 1]] -> 2
[[1, 0, 0], [0, 0, 0], [0, 0, 0]] -> 4
[[1, 1], [1, 1]] -> -1
[[0, 0], [0, 0]] -> -1


## 3. Kiến thức nền

### 3.1 Khoảng cách Manhattan

Là số bước tối thiểu để đi từ ô này sang ô kia nếu chỉ đi **lên/xuống/trái/phải**, không vật cản. Tên gọi từ các đường phố ô bàn cờ ở Manhattan.

In [2]:
def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

print(manhattan((1, 1), (0, 0)), manhattan((2, 2), (0, 0)))
assert manhattan((2, 2), (0, 0)) == 4

2 4


### 3.2 Duyệt 4 hướng trên lưới

Dùng mảng `DIRS` và kiểm tra biên `0 <= r < n`. Đây là khung chung của mọi bài BFS trên ma trận.

In [3]:
DIRS = [(1, 0), (-1, 0), (0, 1), (0, -1)]

def neighbors(r, c, n):
    for dr, dc in DIRS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < n and 0 <= nc < n:
            yield nr, nc

print(list(neighbors(0, 0, 3)), list(neighbors(1, 1, 3)))
assert len(list(neighbors(1, 1, 3))) == 4

[(1, 0), (0, 1)] [(2, 1), (0, 1), (1, 2), (1, 0)]


### 3.3 BFS một nguồn trên lưới

BFS (Breadth-First Search — tìm kiếm theo chiều rộng) thăm ô theo thứ tự **khoảng cách tăng dần** từ nguồn. Đánh dấu `dist` **ngay khi đẩy vào hàng đợi** để không đẩy một ô hai lần.

In [4]:
def bfs_dist(n, src):
    dist = [[-1] * n for _ in range(n)]
    dist[src[0]][src[1]] = 0
    q = deque([src])
    while q:
        r, c = q.popleft()
        for nr, nc in neighbors(r, c, n):
            if dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1
                q.append((nr, nc))
    return dist

for row in bfs_dist(3, (0, 0)):
    print(row)

[0, 1, 2]
[1, 2, 3]
[2, 3, 4]


Nhận xét: trên lưới **không vật cản**, `dist` của BFS 4 hướng **bằng đúng** khoảng cách Manhattan. Kiểm chứng:

In [5]:
n = 5
d = bfs_dist(n, (1, 3))
assert all(d[r][c] == manhattan((r, c), (1, 3)) for r in range(n) for c in range(n))
print("BFS 4 hướng == Manhattan trên lưới trống")

BFS 4 hướng == Manhattan trên lưới trống


### 3.4 BFS đa nguồn (multi-source BFS)

Nếu đẩy **nhiều nguồn** vào hàng đợi với `dist = 0` từ đầu, BFS cho mỗi ô khoảng cách tới **nguồn gần nhất**. Hình dung: thả nhiều viên đá cùng lúc xuống mặt hồ, các gợn sóng lan đều — ô nào bị gợn sóng chạm tới trước là của viên đá gần nhất.

In [6]:
def multi_bfs(n, sources):
    dist = [[-1] * n for _ in range(n)]
    q = deque()
    for r, c in sources:
        dist[r][c] = 0
        q.append((r, c))
    while q:
        r, c = q.popleft()
        for nr, nc in neighbors(r, c, n):
            if dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1
                q.append((nr, nc))
    return dist

for row in multi_bfs(4, [(0, 0), (3, 3)]):
    print(row)

[0, 1, 2, 3]
[1, 2, 3, 2]
[2, 3, 2, 1]
[3, 2, 1, 0]


### 3.5 Sao chép ma trận

Lời giải "ghi đè lên grid" sẽ phá input. Trong notebook ta luôn sao chép `[row[:] for row in grid]` trước khi sửa, để các test sau vẫn dùng được input gốc.

In [7]:
g = [[1, 0], [0, 0]]
g2 = [row[:] for row in g]
g2[0][1] = 9
print(g, g2)
assert g[0][1] == 0

[[1, 0], [0, 0]] [[1, 9], [0, 0]]


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2: đất duy nhất ở `(0,0)`. Khoảng cách tới đất gần nhất của mỗi ô:

```
0 1 2
1 2 3
2 3 4
```

Ô nước xa nhất là `(2,2)` với khoảng cách `4`.

Ví dụ 1 (đất ở bốn góc):

```
0 1 0
1 2 1
0 1 0
```

→ đáp án `2`. Như vậy nếu ta có bảng `near[r][c]` = khoảng cách tới đất gần nhất, đáp án chỉ là `max` của bảng (trên các ô nước). Câu hỏi là tính bảng này nhanh thế nào.

In [8]:
for g, _ in examples[:2]:
    n = len(g)
    lands = [(r, c) for r in range(n) for c in range(n) if g[r][c] == 1]
    for row in multi_bfs(n, lands):
        print(row)
    print()

[0, 1, 0]
[1, 2, 1]
[0, 1, 0]

[0, 1, 2]
[1, 2, 3]
[2, 3, 4]



## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** liệt kê danh sách ô đất và ô nước. Với mỗi ô nước, tính `min` khoảng cách Manhattan tới mọi ô đất; lấy `max`.

In [9]:
def max_distance_brute(grid):
    n = len(grid)
    lands = [(r, c) for r in range(n) for c in range(n) if grid[r][c] == 1]
    waters = [(r, c) for r in range(n) for c in range(n) if grid[r][c] == 0]
    if not lands or not waters:
        return -1
    return max(min(manhattan(w, l) for l in lands) for w in waters)

for g, want in examples:
    got = max_distance_brute(g)
    print(got, want)
    assert got == want

2 2
4 4
-1 -1
-1 -1


**Độ phức tạp:** `#nước × #đất ≤ (n²/2)²` → `O(n^4)`. Với `n = 100`: tới `2.5·10^7` cặp (khi nửa đất nửa nước) — chậm trong Python (vài chục giây), không ổn.

**Nút thắt:** mỗi ô nước tính lại khoảng cách tới **mọi** ô đất, dù chỉ cần ô gần nhất.

## 6. Bước đột phá

> Thay vì hỏi "**mỗi ô nước** gần đất nào nhất?" (n² câu hỏi riêng lẻ), hãy để **tất cả ô đất cùng lan ra** một lượt BFS. Ô nước được chạm tới ở "vòng" thứ `d` thì khoảng cách tới đất gần nhất của nó là `d`.

Invariant của BFS đa nguồn: các ô rời hàng đợi theo thứ tự `dist` **không giảm**; lần đầu một ô được gán `dist`, đó là khoảng cách nhỏ nhất tới **một** nguồn nào đó.

Quy tắc:

- Nếu ô láng giềng chưa được gán `dist` → gán `dist[cha] + 1` và đẩy vào.
- Nếu đã gán → bỏ qua (đã có nguồn gần hơn hoặc bằng chạm tới trước).
- Đáp án = `dist` của ô **cuối cùng** được gán (vì thứ tự không giảm), hoặc `max` của bảng.

Một góc nhìn khác (DP): khoảng cách tới đất gần nhất thoả `near[r][c] = 1 + min(near của 4 láng giềng)` với ô nước. Vì đường đi ngắn nhất Manhattan luôn có thể đi "đơn điệu" theo một trong bốn góc phần tư, hai lượt quét (trên-trái → dưới-phải, rồi ngược lại) là đủ.

In [10]:
# Minh hoạ "vòng lan": số ô được chạm ở mỗi vòng d trong ví dụ 2
g = examples[1][0]
dist = multi_bfs(3, [(0, 0)])
from collections import Counter
print(sorted(Counter(v for row in dist for v in row).items()))

[(0, 1), (1, 2), (2, 3), (3, 2), (4, 1)]


## 7. Các cách tiếp theo

### 7.1 Cách 2 — BFS từ từng ô nước

Với mỗi ô nước, BFS tới khi gặp ô đất đầu tiên. Tốt hơn brute force khi đất dày đặc (gặp sớm), nhưng tệ nhất vẫn `O(n^4)` (một ô đất ở góc). Đây là "cầu nối" để thấy BFS tự nhiên thế nào.

In [11]:
def max_distance_bfs_each(grid):
    n = len(grid)
    has_land = any(1 in row for row in grid)
    has_water = any(0 in row for row in grid)
    if not has_land or not has_water:
        return -1
    best = 0
    for sr in range(n):
        for sc in range(n):
            if grid[sr][sc] == 1:
                continue
            seen = {(sr, sc)}
            q = deque([(sr, sc, 0)])
            while q:
                r, c, d = q.popleft()
                if grid[r][c] == 1:
                    best = max(best, d)
                    break
                for nr, nc in neighbors(r, c, n):
                    if (nr, nc) not in seen:
                        seen.add((nr, nc))
                        q.append((nr, nc, d + 1))
    return best

for g, want in examples:
    assert max_distance_bfs_each(g) == want
print("BFS từng ô OK")

BFS từng ô OK


**Độ phức tạp:** tệ nhất `O(n^2)` lần BFS × `O(n^2)` mỗi lần = `O(n^4)`; bộ nhớ `O(n^2)`.

### 7.2 Cách 3 — BFS đa nguồn (lời giải chuẩn)

Đẩy mọi ô đất vào hàng đợi. Ghi đè trực tiếp lên bản sao lưới: ô nước được thăm thì gán giá trị `dist + 1` (khác 0) để đánh dấu đã thăm. Theo dõi số vòng.

In [12]:
def max_distance_multi_bfs(grid):
    n = len(grid)
    g = [row[:] for row in grid]
    q = deque((r, c) for r in range(n) for c in range(n) if g[r][c] == 1)
    if len(q) == 0 or len(q) == n * n:
        return -1
    d = -1
    while q:
        d += 1
        for _ in range(len(q)):
            r, c = q.popleft()
            for nr, nc in neighbors(r, c, n):
                if g[nr][nc] == 0:
                    g[nr][nc] = 1          # đánh dấu đã thăm ngay khi đẩy vào
                    q.append((nr, nc))
    return d

for g, want in examples:
    assert max_distance_multi_bfs(g) == want
print("multi BFS OK")

multi BFS OK


Giải thích `d`: vòng 0 là các ô đất; vòng `k` là các ô có khoảng cách `k`. Vòng cuối cùng xử lý là khoảng cách lớn nhất. Vì ta tăng `d` ở **đầu** mỗi vòng và khởi tạo `-1`, sau vòng đất `d = 0`, … sau vòng cuối `d` = khoảng cách xa nhất.

**Độ phức tạp:** mỗi ô vào/ra hàng đợi đúng một lần → `O(n^2)` thời gian, `O(n^2)` bộ nhớ.

### 7.3 Cách 4 — Quy hoạch động hai lượt quét

`near[r][c] = 0` nếu đất, ngược lại khởi tạo `INF`.

- Lượt 1 (trên → dưới, trái → phải): `near[r][c] = min(near[r][c], near[r-1][c] + 1, near[r][c-1] + 1)` — xử lý đất nằm phía **trên-trái**.
- Lượt 2 (dưới → trên, phải → trái): `near[r][c] = min(near[r][c], near[r+1][c] + 1, near[r][c+1] + 1)`.

Sau hai lượt, `near` đúng là khoảng cách Manhattan tới đất gần nhất.

In [13]:
def max_distance_dp(grid):
    n = len(grid)
    INF = 2 * n + 5
    near = [[0 if grid[r][c] == 1 else INF for c in range(n)] for r in range(n)]
    for r in range(n):
        for c in range(n):
            if near[r][c]:
                if r > 0:
                    near[r][c] = min(near[r][c], near[r - 1][c] + 1)
                if c > 0:
                    near[r][c] = min(near[r][c], near[r][c - 1] + 1)
    best = -1
    for r in range(n - 1, -1, -1):
        for c in range(n - 1, -1, -1):
            if near[r][c]:
                if r < n - 1:
                    near[r][c] = min(near[r][c], near[r + 1][c] + 1)
                if c < n - 1:
                    near[r][c] = min(near[r][c], near[r][c + 1] + 1)
                best = max(best, near[r][c])
    return -1 if best >= INF else best   # best == -1: không có nước; best >= INF: không có đất

for g, want in examples:
    assert max_distance_dp(g) == want
print("DP OK")

DP OK


**Vì sao hai lượt đủ?** Gọi `(lr, lc)` là đất gần nhất của `(r, c)`. Nếu đất nằm ở góc phần tư trên-trái, có đường ngắn nhất chỉ đi "xuống" và "phải" từ đất tới ô — lượt 1 bắt được. Nếu đất ở dưới-phải — lượt 2. Còn trên-phải hoặc dưới-trái? Lượt 1 lan giá trị theo hàng sang phải, lượt 2 lan sang trái và lên trên; với đất ở **trên-phải**, đường "xuống rồi trái" có phần "xuống" được lượt 1 lan (dọc cột của đất), phần "trái" được lượt 2 lan. Ta kiểm chứng bằng ngẫu nhiên ở mục 12.

**Độ phức tạp:** `O(n^2)` thời gian, `O(n^2)` bộ nhớ (hoặc `O(1)` thêm nếu ghi đè lên lưới).

## 8. TRACE từng bước

Trace BFS đa nguồn trên ví dụ 1. Ký hiệu lưới sau mỗi vòng (`#` = đã thăm, số = vòng ô đó được chạm).

| Vòng d | Ô xử lý | Ô mới được chạm |
|--------|---------|-----------------|
| 0 | 4 góc đất | (0,1),(1,0),(1,2),(2,1) |
| 1 | 4 ô cạnh | (1,1) |
| 2 | (1,1) | — |

Đáp án `d = 2`.

In [14]:
def trace_multi_bfs(grid):
    n = len(grid)
    level = [[0 if grid[r][c] == 1 else None for c in range(n)] for r in range(n)]
    q = deque((r, c) for r in range(n) for c in range(n) if grid[r][c] == 1)
    if len(q) in (0, n * n):
        return -1
    d = -1
    while q:
        d += 1
        frontier = list(q)
        for _ in range(len(q)):
            r, c = q.popleft()
            for nr, nc in neighbors(r, c, n):
                if level[nr][nc] is None:
                    level[nr][nc] = d + 1
                    q.append((nr, nc))
        print(f"vòng d={d}: xử lý {frontier} -> chạm mới {list(q)}")
    for row in level:
        print("   ", row)
    return d

assert trace_multi_bfs(examples[0][0]) == 2
print()
assert trace_multi_bfs([[0, 0, 0, 0], [0, 1, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]]) == 4

vòng d=0: xử lý [(0, 0), (0, 2), (2, 0), (2, 2)] -> chạm mới [(1, 0), (0, 1), (1, 2), (2, 1)]
vòng d=1: xử lý [(1, 0), (0, 1), (1, 2), (2, 1)] -> chạm mới [(1, 1)]
vòng d=2: xử lý [(1, 1)] -> chạm mới []
    [0, 1, 0]
    [1, 2, 1]
    [0, 1, 0]

vòng d=0: xử lý [(1, 1)] -> chạm mới [(2, 1), (0, 1), (1, 2), (1, 0)]
vòng d=1: xử lý [(2, 1), (0, 1), (1, 2), (1, 0)] -> chạm mới [(3, 1), (2, 2), (2, 0), (0, 2), (0, 0), (1, 3)]
vòng d=2: xử lý [(3, 1), (2, 2), (2, 0), (0, 2), (0, 0), (1, 3)] -> chạm mới [(3, 2), (3, 0), (2, 3), (0, 3)]
vòng d=3: xử lý [(3, 2), (3, 0), (2, 3), (0, 3)] -> chạm mới [(3, 3)]
vòng d=4: xử lý [(3, 3)] -> chạm mới []
    [2, 1, 2, 3]
    [1, 0, 1, 2]
    [2, 1, 2, 3]
    [3, 2, 3, 4]


Trace DP hai lượt trên ví dụ 2: in bảng `near` sau mỗi lượt.

In [15]:
def trace_dp(grid):
    n = len(grid)
    INF = 99
    near = [[0 if grid[r][c] else INF for c in range(n)] for r in range(n)]
    for r in range(n):
        for c in range(n):
            if near[r][c]:
                up = near[r - 1][c] + 1 if r else INF
                left = near[r][c - 1] + 1 if c else INF
                near[r][c] = min(near[r][c], up, left)
    print("sau lượt 1:"); [print("   ", row) for row in near]
    for r in range(n - 1, -1, -1):
        for c in range(n - 1, -1, -1):
            if near[r][c]:
                down = near[r + 1][c] + 1 if r < n - 1 else INF
                right = near[r][c + 1] + 1 if c < n - 1 else INF
                near[r][c] = min(near[r][c], down, right)
    print("sau lượt 2:"); [print("   ", row) for row in near]
    return near

nr_ = trace_dp([[0, 0, 0], [0, 0, 0], [0, 0, 1]])

sau lượt 1:
    [99, 99, 99]
    [99, 99, 99]
    [99, 99, 0]
sau lượt 2:
    [4, 3, 2]
    [3, 2, 1]
    [2, 1, 0]


Chú ý ví dụ trên: đất ở **góc dưới-phải**, lượt 1 không lan được gì (tất cả vẫn `99`), lượt 2 lan toàn bộ — đúng với lý luận "mỗi lượt xử lý một hướng".

## 9. Code nộp LeetCode

Chọn BFS đa nguồn — `O(n²)`, trực quan.

In [16]:
from collections import deque
from typing import List


class Solution:
    def maxDistance(self, grid: List[List[int]]) -> int:
        n = len(grid)
        g = [row[:] for row in grid]       # không phá input (trên LeetCode có thể ghi đè trực tiếp)
        q = deque((r, c) for r in range(n) for c in range(n) if g[r][c] == 1)
        if len(q) == 0 or len(q) == n * n:
            return -1
        d = -1
        while q:
            d += 1
            for _ in range(len(q)):
                r, c = q.popleft()
                for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
                    if 0 <= nr < n and 0 <= nc < n and g[nr][nc] == 0:
                        g[nr][nc] = 1
                        q.append((nr, nc))
        return d


sol = Solution()
for g, want in examples:
    assert sol.maxDistance(g) == want
print("Solution OK")

Solution OK


## 10. Vì sao đúng

1. **Trên lưới không vật cản, khoảng cách BFS 4 hướng = Manhattan.** Mỗi bước thay đổi `|Δr| + |Δc|` đúng 1 nên cần ít nhất Manhattan bước; và luôn đi được đúng chừng ấy bước (đi thẳng hàng rồi thẳng cột).
2. **BFS đa nguồn = khoảng cách tới nguồn gần nhất.** Tương đương BFS một nguồn trên đồ thị thêm một "siêu nguồn" ảo nối tới mọi ô đất bằng cạnh độ dài 0 (ta đặt mọi ô đất ở vòng 0). BFS một nguồn cho khoảng cách ngắn nhất; khoảng cách ngắn nhất từ siêu nguồn = `min` qua các ô đất.
3. **Vòng cuối = max.** BFS xử lý theo vòng tăng dần; vòng cuối cùng không rỗng chứa các ô có `near` lớn nhất. Vì có ít nhất một ô nước (đã loại trường hợp toàn đất), vòng cuối ≥ 1 và gồm toàn ô nước.
4. **Trường hợp `-1`:** không đất → không có khoảng cách; không nước → không có ô để chọn. Cả hai được kiểm tra trước.

## 11. Time / Space Complexity

Gọi `N = n²` là số ô.

| Cách | Time | Space |
|------|------|-------|
| 1. Brute mọi cặp | `O(#nước · #đất)` ≤ `O(N²)` | `O(N)` |
| 2. BFS từng ô nước | `O(N²)` tệ nhất | `O(N)` |
| 3. BFS đa nguồn | `O(N)` | `O(N)` |
| 4. DP hai lượt | `O(N)` | `O(N)` (hoặc `O(1)` nếu ghi đè) |

Với `n = 100`: `N = 10^4`, cách 3/4 chạy cỡ vài chục mili-giây.

## 12. Bộ test

### 12.1 Ví dụ + edge case

In [17]:
impls = {
    "brute": max_distance_brute,
    "bfs_each": max_distance_bfs_each,
    "multi_bfs": max_distance_multi_bfs,
    "dp": max_distance_dp,
    "Solution": Solution().maxDistance,
}
edge = [
    ([[0]], -1),
    ([[1]], -1),
    ([[1, 0], [0, 0]], 2),
    ([[0, 0, 0], [0, 1, 0], [0, 0, 0]], 2),
    ([[0, 0, 1], [0, 0, 0], [0, 0, 0]], 4),   # đất ở góc trên-phải (kiểm tra DP)
    ([[0, 0, 0], [0, 0, 0], [1, 0, 0]], 4),   # đất ở góc dưới-trái
    ([[1, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 1]], 3),
]
for name, f in impls.items():
    for g, want in examples + edge:
        before = copy.deepcopy(g)
        assert f(g) == want, (name, g)
        assert g == before, (name, "đã phá input")
print("all impls pass", len(examples + edge))

all impls pass 11


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

Mật độ đất thay đổi từ rất thưa đến rất dày, bao gồm cả lưới toàn đất/toàn nước.

In [18]:
rng = random.Random(1162)
for _ in range(400):
    n = rng.randint(1, 7)
    p = rng.choice([0.0, 0.05, 0.2, 0.5, 0.9, 1.0])
    g = [[1 if rng.random() < p else 0 for _ in range(n)] for _ in range(n)]
    want = max_distance_brute(g)
    for name, f in impls.items():
        assert f(g) == want, (name, g)
print("random cross-check OK")

random cross-check OK


### 12.3 Test giới hạn

`n = 100`: một ô đất ở góc (đáp án `198`), đất ngẫu nhiên 1%, và bàn cờ.

In [19]:
n = 100
cases = []
g = [[0] * n for _ in range(n)]; g[0][0] = 1; cases.append((g, 198))
rng = random.Random(5)
g = [[1 if rng.random() < 0.01 else 0 for _ in range(n)] for _ in range(n)]; cases.append((g, None))
g = [[(r + c) % 2 for c in range(n)] for r in range(n)]; cases.append((g, 1))
for g, want in cases:
    t0 = time.perf_counter()
    a = Solution().maxDistance(g)
    b = max_distance_dp(g)
    assert a == b and (want is None or a == want)
    print(a, f"{time.perf_counter() - t0:.3f}s")

198 0.010s
28 0.010s
1 0.009s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trường hợp toàn nước / toàn đất

In [20]:
def wrong_no_minus_one(grid):
    n = len(grid)
    g = [row[:] for row in grid]
    q = deque((r, c) for r in range(n) for c in range(n) if g[r][c] == 1)
    d = -1
    while q:
        d += 1
        for _ in range(len(q)):
            r, c = q.popleft()
            for nr, nc in neighbors(r, c, n):
                if g[nr][nc] == 0:
                    g[nr][nc] = 1
                    q.append((nr, nc))
    return d

print("toàn đất -> sai:", wrong_no_minus_one([[1, 1], [1, 1]]), "| đúng:", -1)
assert wrong_no_minus_one([[1, 1], [1, 1]]) == 0

toàn đất -> sai: 0 | đúng: -1


**Vì sao sai:** toàn đất thì BFS chạy một vòng (vòng 0), trả `0` thay vì `-1`. (Toàn nước tình cờ trả `-1` vì hàng đợi rỗng.) **Sửa:** kiểm tra `len(q) == n*n` trước.

### Lỗi 2 — Đánh dấu thăm khi **lấy ra** thay vì khi **đẩy vào**

In [21]:
def wrong_mark_on_pop(grid):
    n = len(grid)
    seen = set()
    q = deque((r, c, 0) for r in range(n) for c in range(n) if grid[r][c] == 1)
    if len(q) in (0, n * n):
        return -1
    best, pushes = 0, 0
    while q:
        r, c, d = q.popleft()
        if (r, c) in seen:
            continue
        seen.add((r, c))
        best = max(best, d)
        for nr, nc in neighbors(r, c, n):
            if (nr, nc) not in seen:
                q.append((nr, nc, d + 1)); pushes += 1
    return best, pushes

g = [[0] * 30 for _ in range(30)]; g[0][0] = 1
res, pushes = wrong_mark_on_pop(g)
print("đáp án vẫn đúng:", res, "nhưng số lần push:", pushes, "so với số ô:", 30 * 30)
assert pushes > 30 * 30  # nhiều hơn số ô: có ô bị đẩy vào hàng đợi nhiều lần

đáp án vẫn đúng: 58 nhưng số lần push: 1740 so với số ô: 900


**Vì sao sai (về hiệu năng):** một ô bị đẩy vào hàng đợi nhiều lần (mỗi láng giềng đẩy một lần) → bộ nhớ và thời gian tăng đáng kể (ở đây gần gấp đôi số ô); với biến thể đếm số lần thăm còn sai cả kết quả. **Sửa:** đánh dấu ngay khi đẩy (code mục 9).

### Lỗi 3 — DP chỉ quét một lượt

In [22]:
def wrong_dp_one_pass(grid):
    n = len(grid)
    INF = 10**9
    near = [[0 if grid[r][c] else INF for c in range(n)] for r in range(n)]
    for r in range(n):
        for c in range(n):
            if near[r][c]:
                if r: near[r][c] = min(near[r][c], near[r - 1][c] + 1)
                if c: near[r][c] = min(near[r][c], near[r][c - 1] + 1)
    vals = [near[r][c] for r in range(n) for c in range(n) if grid[r][c] == 0]
    return max(vals) if vals and any(1 in row for row in grid) else -1

g = [[0, 0, 0], [0, 0, 0], [0, 0, 1]]
print("sai:", wrong_dp_one_pass(g), "| đúng:", max_distance_dp(g))
assert wrong_dp_one_pass(g) != max_distance_dp(g)

sai: 1000000000 | đúng: 4


**Vì sao sai:** lượt trên-trái → dưới-phải chỉ "nhìn thấy" đất ở phía trên/trái; đất ở dưới-phải bị bỏ qua và giá trị kẹt ở `INF`. **Sửa:** thêm lượt ngược.

### Lỗi 4 — Trả về khoảng cách tới đất **xa nhất** thay vì gần nhất

In [23]:
def wrong_max_max(grid):
    n = len(grid)
    lands = [(r, c) for r in range(n) for c in range(n) if grid[r][c]]
    waters = [(r, c) for r in range(n) for c in range(n) if not grid[r][c]]
    if not lands or not waters:
        return -1
    return max(max(manhattan(w, l) for l in lands) for w in waters)

g = examples[0][0]
print("sai:", wrong_max_max(g), "| đúng:", max_distance_brute(g))
assert wrong_max_max(g) != max_distance_brute(g)

sai: 3 | đúng: 2


**Vì sao sai:** đề là "max của **min**": mỗi ô nước tính tới đất **gần nhất**. **Sửa:** `max(min(...))`.

## 14. Mẹo & nhận dạng pattern

- **"Khoảng cách tới X gần nhất" cho mọi ô** → BFS đa nguồn từ tất cả X (LC 542 01 Matrix, LC 994 Rotting Oranges, LC 286 Walls and Gates).
- **"Max của min khoảng cách"** → tính bảng khoảng cách gần nhất rồi lấy max; với BFS, chính là số vòng cuối.
- **Lưới không vật cản + Manhattan** → có thể dùng DP hai lượt, không cần hàng đợi.
- Có vật cản → chỉ BFS dùng được (DP hai lượt sai vì đường ngắn nhất có thể phải vòng).
- Đánh dấu thăm **khi đẩy vào**.

## 15. Biến thể

1. **LC 542 01 Matrix** — trả về toàn bộ bảng khoảng cách tới ô `0` gần nhất.
2. **LC 994 Rotting Oranges** — BFS đa nguồn đếm số phút; có ô không bị ảnh hưởng → `-1`.
3. **LC 1765 Map of Highest Peak** — gán độ cao = khoảng cách tới nước gần nhất.
4. **Khoảng cách Chebyshev (8 hướng)** — BFS 8 hướng; DP hai lượt cần thêm ô chéo.
5. **Có vật cản** — BFS bỏ qua ô vật cản; DP không còn đúng.

In [24]:
# Biến thể 4: khoảng cách Chebyshev (đi được 8 hướng)
def max_distance_chebyshev(grid):
    n = len(grid)
    dist = [[-1] * n for _ in range(n)]
    q = deque()
    for r in range(n):
        for c in range(n):
            if grid[r][c]:
                dist[r][c] = 0; q.append((r, c))
    if len(q) in (0, n * n):
        return -1
    best = 0
    while q:
        r, c = q.popleft()
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                nr, nc = r + dr, c + dc
                if 0 <= nr < n and 0 <= nc < n and dist[nr][nc] == -1:
                    dist[nr][nc] = dist[r][c] + 1
                    best = max(best, dist[nr][nc])
                    q.append((nr, nc))
    return best

print(max_distance_chebyshev(examples[1][0]), max_distance_chebyshev(examples[0][0]))
assert max_distance_chebyshev(examples[1][0]) == 2

2 1


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Brute | Mọi cặp (nước, đất) | `O(N²)` | `O(N)` | Kiểm chứng |
| 2. BFS từng ô nước | Mỗi ô tìm đất gần nhất | `O(N²)` | `O(N)` | Minh hoạ, đất dày |
| 3. BFS đa nguồn | Mọi đất lan cùng lúc | `O(N)` | `O(N)` | Chuẩn, tổng quát (có vật cản vẫn dùng được) |
| 4. DP hai lượt | `near = 1 + min(láng giềng)`, 2 hướng quét | `O(N)` | `O(N)`/`O(1)` | Lưới không vật cản, không cần hàng đợi |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `[[0,0,0],[0,0,0],[0,0,0]]` → ? và `[[0,1],[1,0]]` → ?

**Bài 2 (tính tay):** lập bảng `near` cho `[[0,0,0,0],[0,0,0,0],[0,0,1,0],[0,0,0,0]]` và tìm đáp án.

**Bài 3 (điền TODO):** viết hàm trả về **toàn bộ** bảng khoảng cách tới đất gần nhất (LC 542 phiên bản đảo).

**Bài 4 (tự viết lại):** viết lại DP hai lượt **ghi đè trực tiếp** lên bản sao lưới (không tạo `near` riêng), lưu ý đất phải là 0 khoảng cách.

**Bài 5 (mini challenge):** lưới có thêm ô `2` là **vật cản** (không đi qua). Trả về max khoảng cách BFS từ ô nước tới đất gần nhất; ô nước không tới được đất bị bỏ qua; nếu không có ô nước tới được thì `-1`.

In [25]:
# Bài 3 — TODO
def nearest_land_table(grid):
    # TODO: BFS đa nguồn, trả về ma trận dist
    raise NotImplementedError

# assert nearest_land_table([[1,0,0],[0,0,0],[0,0,0]]) == [[0,1,2],[1,2,3],[2,3,4]]

In [26]:
# Bài 5 — TODO
def max_distance_with_walls(grid):
    # TODO: 0 = nước, 1 = đất, 2 = vật cản
    pass

# assert max_distance_with_walls([[1,2,0],[0,2,0],[0,0,0]]) == 6

## 18. Gợi ý 3 mức

- **Bài 1:** Mức 1: có đất không? Mức 2: không đất → `-1`. Mức 3: `[[0,1],[1,0]]` mọi nước kề đất → `1`.
- **Bài 2:** Mức 1: đất ở `(2,2)`. Mức 2: `near = |r-2| + |c-2|`. Mức 3: xa nhất là `(0,0)` → `4`.
- **Bài 3:** Mức 1: copy `multi_bfs`. Mức 2: nguồn là ô `1`. Mức 3: trả `dist`.
- **Bài 4:** Mức 1: đổi đất thành `0`, nước thành `INF`. Mức 2: hai lượt như cũ. Mức 3: max trên ô ban đầu là nước.
- **Bài 5:** Mức 1: BFS đa nguồn bỏ qua ô `2`. Mức 2: ô nước không chạm được giữ `-1`. Mức 3: đáp án = max của `dist` trên ô nước đã chạm, hoặc `-1`.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
assert Solution().maxDistance([[0, 0, 0], [0, 0, 0], [0, 0, 0]]) == -1
assert Solution().maxDistance([[0, 1], [1, 0]]) == 1
# Bài 2
assert Solution().maxDistance([[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 1, 0], [0, 0, 0, 0]]) == 4

# Bài 3
def nearest_land_table_sol(grid):
    n = len(grid)
    return multi_bfs(n, [(r, c) for r in range(n) for c in range(n) if grid[r][c] == 1])

assert nearest_land_table_sol([[1, 0, 0], [0, 0, 0], [0, 0, 0]]) == [[0, 1, 2], [1, 2, 3], [2, 3, 4]]

# Bài 4
def max_distance_dp_inplace(grid):
    n = len(grid)
    INF = 2 * n + 5
    g = [[0 if v else INF for v in row] for row in grid]
    for r in range(n):
        for c in range(n):
            if r: g[r][c] = min(g[r][c], g[r - 1][c] + 1)
            if c: g[r][c] = min(g[r][c], g[r][c - 1] + 1)
    for r in range(n - 1, -1, -1):
        for c in range(n - 1, -1, -1):
            if r < n - 1: g[r][c] = min(g[r][c], g[r + 1][c] + 1)
            if c < n - 1: g[r][c] = min(g[r][c], g[r][c + 1] + 1)
    best = max(v for row in g for v in row)
    return -1 if best == 0 or best >= INF else best

rng = random.Random(19)
for _ in range(200):
    n = rng.randint(1, 6)
    g = [[int(rng.random() < 0.2) for _ in range(n)] for _ in range(n)]
    assert max_distance_dp_inplace(g) == max_distance_brute(g)
print("bài 1-4 OK")

bài 1-4 OK


In [28]:
# Bài 5
def max_distance_with_walls_sol(grid):
    n = len(grid)
    dist = [[-1] * n for _ in range(n)]
    q = deque()
    for r in range(n):
        for c in range(n):
            if grid[r][c] == 1:
                dist[r][c] = 0; q.append((r, c))
    best = -1
    while q:
        r, c = q.popleft()
        for nr, nc in neighbors(r, c, n):
            if grid[nr][nc] == 0 and dist[nr][nc] == -1:
                dist[nr][nc] = dist[r][c] + 1
                best = max(best, dist[nr][nc])
                q.append((nr, nc))
    return best

assert max_distance_with_walls_sol([[1, 2, 0], [0, 2, 0], [0, 0, 0]]) == 6
assert max_distance_with_walls_sol([[1, 2], [2, 0]]) == -1
print("bài 5 OK")

bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
q = deque((r, c) for r in range(n) for c in range(n) if grid[r][c] == 1)
if len(q) in (0, n * n): return -1
d = -1
while q:
    d += 1
    for _ in range(len(q)):
        r, c = q.popleft()
        for nr, nc in ((r+1,c),(r-1,c),(r,c+1),(r,c-1)):
            if 0 <= nr < n and 0 <= nc < n and grid[nr][nc] == 0:
                grid[nr][nc] = 1          # đánh dấu khi đẩy
                q.append((nr, nc))
return d
```

**Checklist:**

- [ ] Kiểm tra toàn đất **và** toàn nước → `-1`?
- [ ] Đẩy **tất cả** ô đất vào hàng đợi trước khi BFS?
- [ ] Đánh dấu thăm khi đẩy vào?
- [ ] `d` khởi tạo `-1` và tăng ở đầu mỗi vòng?
- [ ] Kiểm tra biên `0 <= nr < n`?
- [ ] Nếu dùng DP: đủ **hai** lượt quét?